In [2]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import joblib

# Load dataset
df = pd.read_csv('Indonesian_Food_Recipes_Cleaned.csv')

# Pastikan tidak ada nilai kosong di kolom Ingredients Cleaned
df['Ingredients Cleaned'] = df['Ingredients Cleaned'].fillna('')

In [3]:
# Inisialisasi TF-IDF Vectorizer
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))

# Transformasi kolom Ingredients Cleaned menjadi matriks TF-IDF
tfidf_matrix = vectorizer.fit_transform(df['Ingredients Cleaned'])

print("Matriks TF-IDF berhasil dibuat dengan bentuk:", tfidf_matrix.shape)

Matriks TF-IDF berhasil dibuat dengan bentuk: (12365, 5000)


In [4]:
def calculate_spi(days_remaining, alpha=2.0):
    """
    Menghitung skor urgensi berdasarkan sisa hari (d).
    Rumus: 1 / (d + 1)^alpha
    """
    return 1 / ((days_remaining + 1) ** alpha)

In [5]:
def get_recommendations(user_ingredients, inventory_expiry, top_k=10):
    """
    user_ingredients: list bahan (contoh: ['bawang putih', 'bayam', 'ayam'])
    inventory_expiry: dict sisa hari (contoh: {'bayam': 1, 'ayam': 5})
    """
    
    # 1. Hitung Cosine Similarity antara input user dan database resep
    user_input_text = ' '.join(user_ingredients)
    user_vector = vectorizer.transform([user_input_text])
    cos_sim = cosine_similarity(user_vector, tfidf_matrix).flatten()
    
    # 2. Tambahkan Skor SPI
    # Kita akan membuat array SPI dengan ukuran yang sama dengan jumlah resep
    spi_scores = np.zeros(len(df))
    
    for ingredient, days in inventory_expiry.items():
        urgency_score = calculate_spi(days)
        
        # Cari resep yang mengandung bahan kritis tersebut
        # Kita cek di kolom 'Ingredients Cleaned'
        mask = df['Ingredients Cleaned'].str.contains(ingredient, case=False)
        spi_scores[mask] += urgency_score
    
    # 3. Final Scoring (Similarity + SPI)
    # Bobot w1 dan w2 bisa disesuaikan (Contoh: 0.6 dan 0.4 sesuai Tabel 4.14)
    w1 = 0.6
    w2 = 0.4
    final_scores = (cos_sim * w1) + (spi_scores * w2)
    
    # 4. Ambil Top-K Resep
    top_indices = final_scores.argsort()[-top_k:][::-1]
    
    results = df.iloc[top_indices].copy()
    results['final_score'] = final_scores[top_indices]
    
    return results[['Title', 'Ingredients Cleaned', 'final_score']]

In [6]:
# Input simulasi dari aplikasi nanti
bahan_saya = ['bayam', 'ayam', 'bawang putih']
sisa_hari = {'bayam': 1, 'ayam': 5}

# Jalankan rekomendasi
rekomendasi = get_recommendations(bahan_saya, sisa_hari)
print(rekomendasi)

                                                   Title  \
7100                                    Bayam Orek Telur   
4221                             Sayur bayam dan tongkol   
7271                                   Dadar Telur Bayam   
11930        Diet GM Day 3: Sayur Bayam dan Sambal Tempe   
9171                   Sayur bening (bayam, timun, tahu)   
6304                       Sayur Bayam Tahu dapursabilla   
7343                                   Oseng bayam Telur   
4734                               Telur dadar isi bayam   
10557                     Sayur bayam jagung tahu bening   
6687   Salmon panggang oven dengan bayam dan jagung m...   

                                     Ingredients Cleaned  final_score  
7100   genggam bayam , telur , bawang putih , bawang ...     0.372326  
4221   bayam , jagung , tempe , tongkol , bawang mera...     0.369494  
7271   telur , bawang merah , rajang , daun bawang , ...     0.350767  
11930  sayur bayam , bayam , bawang merah , bawang 

In [7]:
# Simpan matriks, vectorizer, dan dataframe
joblib.dump(vectorizer, 'tfidf_vectorizer.pkl')
joblib.dump(tfidf_matrix, 'recipe_matrix.pkl')
df.to_pickle('recipe_data.pkl')

print("Model AI NirSisa berhasil disimpan!")

Model AI NirSisa berhasil disimpan!
